Starting from each PubChem CID, we retrieve its interaction and pathway information to identify the corresponding target PDB structures. These PDBs serve as the basis for BLAST searches to find malaria proteins that are structurally or functionally similar to the drug targets derived from the CDOT dataset.

In [ ]:
import os
import pandas as pd
import requests

# CONFIGURATION

CDOT_50_PATH = '/content/drive/MyDrive/Drug Repurposing Project/cdot_50.csv'
CDOT_70_PATH = '/content/drive/MyDrive/Drug Repurposing Project/cdot_70.csv'
COMBINED_OUTPUT_PATH = '/content/drive/MyDrive/cdot_50_70.csv'
INTERACTIONS_FOLDER = '/content/drive/MyDrive/Drug Repurposing Project/Interactions_Results'

# STEP 1: Load and combine CDOT datasets
def load_and_combine_cdot_data(path1, path2, output_path):
    df1 = pd.read_csv(path1)
    df2 = pd.read_csv(path2)
    combined_df = pd.concat([df1, df2], ignore_index=True)
    combined_df.to_csv(output_path, index=False)
    return combined_df

# STEP 2: Clean up and extract necessary columns
def preprocess_combined_df(file_path):
    df = pd.read_csv(file_path)
    df = df[['pubchem_cid', 'target', 'smiles']].dropna(subset=['pubchem_cid'])
    df['pubchem_cid'] = df['pubchem_cid'].astype(int)
    return df

# STEP 3: Download pathway interaction CSVs from PubChem
def fetch_pathways_results(pubchem_ids, folder_path):
    os.makedirs(folder_path, exist_ok=True)

    for cid in pubchem_ids:
        url = (
            f"https://pubchem.ncbi.nlm.nih.gov/sdq/sdqagent.cgi?infmt=json&outfmt=csv&query="
            f"{{\"download\":\"*\",\"collection\":\"pdb\",\"order\":[\"resolution,asc\"],"
            f"\"start\":1,\"limit\":10000000,\"downloadfilename\":\"pubchem_cid_{cid}_pdb\","
            f"\"where\":{{\"ands\":[{{\"cid\":\"{cid}\"}}]}}}}"
        )
        response = requests.get(url)
        file_path = os.path.join(folder_path, f"{cid}.csv")

        if response.status_code == 200:
            with open(file_path, 'wb') as file:
                file.write(response.content)
            print(f"[✓] Saved: {file_path}")
        else:
            print(f"[✗] Failed for CID {cid} — Status Code: {response.status_code}")

# MAIN EXECUTION
if __name__ == "__main__":
    combined_df = load_and_combine_cdot_data(CDOT_50_PATH, CDOT_70_PATH, COMBINED_OUTPUT_PATH)
    cleaned_df = preprocess_combined_df(COMBINED_OUTPUT_PATH)
    pubchem_ids = cleaned_df['pubchem_cid'].tolist()
    fetch_pathways_results(pubchem_ids, INTERACTIONS_FOLDER)


In [ ]:
import os
import pandas as pd


# CONFIGURATION

INTERACTIONS_FOLDER = '/content/drive/MyDrive/Drug Repurposing Project/Interactions_Results'
CDOT_FILE = '/content/drive/MyDrive/cdot_50_70.csv'
INTERACTIONS_OUTPUT = '/content/drive/MyDrive/Drug Repurposing Project/pubchem_target.csv'
FINAL_OUTPUT = '/content/drive/MyDrive/Drug Repurposing Project/for_blast.csv'


# STEP 1: Load all non-empty CSVs from folder and append pubchem_cid
def load_interactions(folder_path):
    all_dfs = []
    empty_csvs = []

    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        if not file.endswith('.csv') or os.path.isdir(file_path):
            continue
        try:
            df = pd.read_csv(file_path)
            if df.empty:
                empty_csvs.append(file)
                continue

            pubchem_id = int(os.path.splitext(file)[0])
            df['pubchem_cid'] = pubchem_id
            all_dfs.append(df)

        except Exception as e:
            print(f"[✗] Error reading {file}: {e}")
            empty_csvs.append(file)

    print("[✓] Loaded DataFrames:", len(all_dfs))
    print("[!] Empty CSVs:", empty_csvs)
    return all_dfs

# STEP 2: Merge with CDOT to add SMILES and save cleaned file
def merge_with_smiles_and_export(interactions_df, cdot_df, output_path):
    merged = interactions_df.merge(cdot_df[['pubchem_cid', 'smiles']], on='pubchem_cid', how='left')
    final_df = merged[['pdbid', 'lignme', 'pubchem_cid', 'smiles']]
    final_df.to_csv(output_path, index=False)
    print(f"[✓] Saved merged file to: {output_path}")
    return final_df

# MAIN EXECUTION
if __name__ == "__main__":
    # Step 1: Load interaction results and combine
    dfs = load_interactions(INTERACTIONS_FOLDER)
    if dfs:
        combined_df = pd.concat(dfs, ignore_index=True)
        combined_df.to_csv(INTERACTIONS_OUTPUT, index=False)
    else:
        combined_df = pd.DataFrame()

    # Step 2: Merge with cdot smiles
    if not combined_df.empty:
        cdot_df = pd.read_csv(CDOT_FILE)
        df_for_blast = merge_with_smiles_and_export(combined_df, cdot_df, FINAL_OUTPUT)
        print(df_for_blast.head(1))  # Preview first row
    else:
        print("[!] No data to process for SMILES merge.")

    # Step 3: Create an empty dataframe structure for final merged results
    columns = [
        'pubchem_cid', 'target', 'target_id', 'malaria', 'malaria_id',
        'target_ligand', 'malaria_ligand', 'target_gene', 'malaria_gene',
        'blast_evalue', 'blast_len', '3d_evalue', '3d_len',
        'smiles', 'target_seq', 'malaria_seq'
    ]
    ultimate_df = pd.DataFrame(columns=columns)
    print("Initialized `ultimate_df` structure.")
    display(ultimate_df.head())


### Step 2: download all the .fasta files for the proteins to do the Blast

In [ ]:
import os
import pandas as pd
import requests

# CONFIGURATION
CSV_PATH = '/content/drive/MyDrive/Drug Repurposing Project/pubchem_target.csv'
FASTA_FOLDER = '/content/drive/MyDrive/Drug Repurposing Project/fasta_files'

# UTILITY FUNCTION: Download a single FASTA file
def download_fasta(pdb_id: str, save_dir: str) -> None:
    """Download FASTA file from RCSB and save it if not already present."""
    fasta_filename = f"{pdb_id}.fasta"
    fasta_path = os.path.join(save_dir, fasta_filename)

    if os.path.exists(fasta_path):
        print(f"[–] Skipped: {fasta_filename} already exists.")
        return

    url = f"https://www.rcsb.org/fasta/entry/{pdb_id}"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            with open(fasta_path, 'wb') as f:
                f.write(response.content)
            print(f"[✓] Downloaded: {fasta_filename}")
        else:
            print(f"[✗] Failed: {pdb_id} (HTTP {response.status_code})")
    except Exception as e:
        print(f"[!] Error downloading {pdb_id}: {e}")

# MAIN EXECUTION
def main():
    os.makedirs(FASTA_FOLDER, exist_ok=True)
    df = pd.read_csv(CSV_PATH)

    pdb_ids = df['pdbid'].dropna().unique()
    for pdb_id in pdb_ids:
        download_fasta(pdb_id, FASTA_FOLDER)

if __name__ == "__main__":
    main()


7NFD.fasta already exists.
7R2G.fasta already exists.
6GH9.fasta already exists.
4G0V.fasta already exists.
4I41.fasta already exists.
2FUM.fasta already exists.
6VXI.fasta already exists.
2KGP.fasta already exists.
3TVB.fasta already exists.
427D.fasta already exists.
1O0K.fasta already exists.
1D11.fasta already exists.
152D.fasta already exists.
2D34.fasta already exists.
1D10.fasta already exists.
1D33.fasta already exists.
1DA0.fasta already exists.
1JO2.fasta already exists.
308D.fasta already exists.
1VTH.fasta already exists.
1VTI.fasta already exists.
110D.fasta already exists.
7QZ6.fasta already exists.
3F8F.fasta already exists.
7QZ7.fasta already exists.
7QZ8.fasta already exists.
4KR8.fasta already exists.
2DES.fasta already exists.
215D.fasta already exists.
234D.fasta already exists.
235D.fasta already exists.
6FGC.fasta already exists.
6YK1.fasta already exists.
8G6J.fasta already exists.
8A3D.fasta already exists.
3G6E.fasta already exists.
6QZP.fasta already exists.
8

In [ ]:
#first mix cdot_50.csv and cdot_70.csv
"""
df1 = pd.read_csv('/content/drive/MyDrive/Drug Repurposing Project/cdot_50.csv')
df2 = pd.read_csv('/content/drive/MyDrive/Drug Repurposing Project/cdot_70.csv')
df = pd.concat([df1, df2], ignore_index=True)
df.to_csv('/content/cdot_total.csv', index = False)


df1 = pd.read_csv('/content/cdot_total.csv')
df2 = pd.read_csv('/content/target_malaria_ligand_gene_chain_id.csv')

print(df1.columns)
print(df2.columns)


# Ensure both ID columns are of the same type (usually string is safest)
df1['pubchem_cid'] = df1['pubchem_cid'].astype(str)
df2['pubchem_id'] = df2['pubchem_id'].astype(str)

# Create a mapping dictionary from df1
cid_to_smiles = dict(zip(df1['pubchem_cid'], df1['smiles']))

# Apply the mapping to df2
df2['smiles'] = df2['pubchem_id'].map(cid_to_smiles)
"""

In [ ]:
#make a ligand_id column for the drug that we have in mind: we can change the drug in the future too if we wanna focus on lets say the second nonpolymer entity in the list
"""
import pandas as pd

import ast  # for safely evaluating list-like strings

# Load the dataset
df = pd.read_csv('/content/Ultimate_Dataset.csv')

# Add ligand_id as the first ligand in T-lig
def extract_first_ligand(entry):
    if pd.isna(entry) or entry.strip() == '':
        return None
    try:
        # Try to interpret it as a list string
        lig_list = ast.literal_eval(entry) if isinstance(entry, str) else entry
        if isinstance(lig_list, list) and len(lig_list) > 0:
            return lig_list[0]
    except Exception:
        pass
    # If it's a simple comma-separated string
    return entry.split(',')[0].strip()

df['ligand_id'] = df['T-lig-code'].apply(extract_first_ligand)
